In [13]:
pip install pandas scikit-learn notebook ipykernel

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import pandas as pd
import re
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer

Load data

In [15]:
df = pd.read_csv("../data/processed/merged_dataset.csv")

print(df.columns)
df.head()

Index(['name', 'category', 'rating', 'address', 'type'], dtype='str')


,name,category,rating,address,type
0,Taman Budaya Yogyakarta,pusat kebudayaan,4.6,"Jl. Sriwedani No.1, Ngupasan, Kec. Gondomanan,...",wisata
1,Vredeburg Fort Museum,museums,4.7,"Jl. Margo Mulyo No.6, Ngupasan, Kec. Gondomana...",wisata
2,Situs Pesanggrahan Rejawinangun,attractions,4.4,"Jl. Veteran No.77, Warungboto, Kec. Umbulharjo...",wisata
3,Taman Sari Tourist Village,attractions,4.6,"Patehan, Kraton, Yogyakarta City, Special Regi...",wisata
4,Bentara Budaya Yogyakarta (BBY),attractions,4.6,"Jl. Suroto No.2, Kotabaru, Kec. Gondokusuman, ...",wisata


In [16]:
# OPTIONAL
if "subtypes" in df.columns:
    df["subtypes"] = df["subtypes"].fillna("")
else:
    df["subtypes"] = ""

In [17]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)  # hapus simbol
    text = re.sub(r'\s+', ' ', text).strip()     # hapus spasi berlebih
    return text

In [18]:
def clean_subtypes(text):
    text = str(text).lower()
    
    # buang kata terlalu umum
    remove_words = ["attractions", "place", "point", "location"]
    
    words = text.split()
    words = [w for w in words if w not in remove_words]
    
    return " ".join(words)

df["subtypes_clean"] = df["subtypes"].apply(clean_subtypes)

df[["subtypes", "subtypes_clean"]].head()

,subtypes,subtypes_clean
0,,
1,,
2,,
3,,
4,,


In [19]:
df["text"] = (
    df["name"].fillna("") + " " +
    df["category"].fillna("") + " " +
    df["type"].fillna("") + " " +
    df["subtypes"].fillna("")
)

df["text"] = df["text"].apply(clean_text)

df[["name", "text"]].head()

,name,text
0,Taman Budaya Yogyakarta,taman budaya yogyakarta pusat kebudayaan wisata
1,Vredeburg Fort Museum,vredeburg fort museum museums wisata
2,Situs Pesanggrahan Rejawinangun,situs pesanggrahan rejawinangun attractions wi...
3,Taman Sari Tourist Village,taman sari tourist village attractions wisata
4,Bentara Budaya Yogyakarta (BBY),bentara budaya yogyakarta bby attractions wisata


TF-IDF

In [20]:
vectorizer = TfidfVectorizer(
    stop_words="english",   # buang kata ga penting
    ngram_range=(1,2),      # tangkap "hotel murah"
    min_df=2                # buang kata terlalu jarang (noise)
)

tfidf_matrix = vectorizer.fit_transform(df["text"])

print("TF-IDF shape:", tfidf_matrix.shape)

TF-IDF shape: (726, 469)


SAVE MODEL

In [21]:
with open("../data/processed/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

with open("../data/processed/tfidf_matrix.pkl", "wb") as f:
    pickle.dump(tfidf_matrix, f)

print("TF-IDF saved ✅")

TF-IDF saved ✅


In [22]:
# CEK SHAPE
print("Shape:", tfidf_matrix.shape)

# CEK ISI MATRIX (baris pertama)
print(tfidf_matrix[0].toarray())

# CEK FEATURE (kata yang dipelajari model)
print(vectorizer.get_feature_names_out()[:20])

Shape: (726, 469)
[[0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.43016893 0.46691811
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0

In [23]:
feature_names = vectorizer.get_feature_names_out()

print(feature_names[:20])

['1o1' 'adventure' 'adventure attractions' 'agen' 'air' 'air terjun'
 'alam' 'alam wisata' 'ambarrukmo' 'ambarrukmo yogyakarta' 'angkringan'
 'arah' 'area' 'area rekreasi' 'arum' 'asih' 'asli' 'asri' 'attractions'
 'attractions hotel']
